##### add_months()
- is used for **date** manipulation by **adding** a specific **number of months** to a **date column**.

**Syntax**

     add_months(start_date, months_to_add)

|  Argument        |   Description          |
|------------------|------------------------|
| **start_date**   | A column containing **dates** (must be a **DateType or TimestampType or a string** that can be **cast** to one of these types). If it is a **string**, you may need to explicitly **convert** it to a **date type** using **to_date()** for correct behavior. |
| **months_to_add** | An `integer` or a column of integers representing the **number of months to add** (can be `negative to subtract months`). |

**Returns:**
- The **date** that is **months after start**.
- If **months is a negative** value then these **amount of months** will be **deducted from the start**.

| Requirement     | Function                |
| --------------- | ----------------------- |
| Add 1 year      | `add_months(date, 12)`  |
| Subtract 1 year | `add_months(date, -12)` |
| Add 365 days    | `date_add(date, 365)`   |

##### 1) How to get date After One Year?

In [0]:
from pyspark.sql.functions import col, current_date, current_timestamp, add_months

In [0]:
# Adds 12 months to today’s date
df_after = spark.range(1) \
                .select(current_date().alias("today"),
                        add_months(current_date(), 12).alias("after_one_year"),
                        add_months(current_date(), -6).alias("before_6_months"),
                        add_months(current_date(), 3).alias("after_3_months")
                        )

display(df_after)

today,after_one_year,before_6_months,after_3_months
2026-02-14,2027-02-14,2025-08-14,2026-05-14


In [0]:
# Adds 12 months to today’s date time
df_after_ts = spark.range(1) \
                .select(current_timestamp().alias("today"),
                        add_months(current_timestamp(), 12).alias("after_one_year"),
                        add_months(current_timestamp(), -6).alias("before_6_months"),
                        add_months(current_timestamp(), 3).alias("after_3_months")
                        )

display(df_after_ts)

today,after_one_year,before_6_months,after_3_months
2026-03-06T18:36:43.616Z,2027-03-06,2025-09-06,2026-06-06


##### 2) How to Get Date Before One Year?
- `Negative value subtracts months`.

In [0]:
# -12 -> subtracts one year.
df_before = spark.range(1) \
                 .select(current_date().alias("today"),
                         add_months(current_date(), -12).alias("before_one_year"),
                         add_months(current_date(), -6).alias("before_6_months")
                 )

display(df_before)

today,before_one_year,before_6_months
2026-02-14,2025-02-14,2025-08-14


##### 3) If You Have a Date Column

In [0]:
# define data
data = [['Sowmya', 29, 'Chennai', '2020-10-25'],
        ['Bole', 32, 'Bangalore', '2013-10-11'],
        ['Chandini', 35, 'Hyderabad', '2015-10-17'],
        ['Deepthi', 40, 'Nasik', '2022-12-21'],
        ['Swapna', 37, 'Mumbai', '2021-04-14'],
        ['Tharun', 25, 'Delhi', '2021-06-26']] 
  
# define column names
columns = ['emp_name', 'Age', 'City', 'start_date'] 
  
# create dataframe using data and column names
df_dt = spark.createDataFrame(data, columns) \
        .withColumn("start_date", col("start_date").cast("date"))
  
# view dataframe
display(df_dt)

df_dt.printSchema()

emp_name,Age,City,start_date
Sowmya,29,Chennai,2020-10-25
Bole,32,Bangalore,2013-10-11
Chandini,35,Hyderabad,2015-10-17
Deepthi,40,Nasik,2022-12-21
Swapna,37,Mumbai,2021-04-14
Tharun,25,Delhi,2021-06-26


root
 |-- emp_name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- City: string (nullable = true)
 |-- start_date: date (nullable = true)



In [0]:
df_dt_add = (df_dt
  # add a new column that adds 12 months to each date in the "start_date" column
  .withColumn("next_year_date", add_months("start_date", 12))
  # add a new column that adds 3 months to each date in the "start_date" column
  .withColumn("after_3_months", add_months("start_date", 3))
  # add a new column that subtract 12 months to each date in the "start_date" column
  .withColumn("before_one_year", add_months("start_date", -12))
  # add a new column that subtract 6 months to each date in the "start_date" column
  .withColumn("before_6_months", add_months("start_date", -6)))

display(df_dt_add)

emp_name,Age,City,start_date,next_year_date,after_3_months,before_one_year,before_6_months
Sowmya,29,Chennai,2020-10-25,2021-10-25,2021-01-25,2019-10-25,2020-04-25
Bole,32,Bangalore,2013-10-11,2014-10-11,2014-01-11,2012-10-11,2013-04-11
Chandini,35,Hyderabad,2015-10-17,2016-10-17,2016-01-17,2014-10-17,2015-04-17
Deepthi,40,Nasik,2022-12-21,2023-12-21,2023-03-21,2021-12-21,2022-06-21
Swapna,37,Mumbai,2021-04-14,2022-04-14,2021-07-14,2020-04-14,2020-10-14
Tharun,25,Delhi,2021-06-26,2022-06-26,2021-09-26,2020-06-26,2020-12-26


In [0]:
# define data
data = [['Sowmya', 29, 'Chennai', '2020-10-25 00:00:00'],
        ['Bole', 32, 'Bangalore', '2013-10-11 00:00:00'],
        ['Chandini', 35, 'Hyderabad', '2015-10-17 00:00:00'],
        ['Deepthi', 40, 'Nasik', '2022-12-21 00:00:00'],
        ['Swapna', 37, 'Mumbai', '2021-04-14 00:00:00'],
        ['Tharun', 25, 'Delhi', '2021-06-26 00:00:00']] 
  
# define column names
columns = ['emp_name', 'Age', 'City', 'start_time'] 
  
# create dataframe using data and column names
df_dt_ts = spark.createDataFrame(data, columns) \
        .withColumn("start_time", col("start_time").cast("timestamp"))
  
# view dataframe
display(df_dt_ts)

df_dt_ts.printSchema()

emp_name,Age,City,start_time
Sowmya,29,Chennai,2020-10-25T00:00:00.000Z
Bole,32,Bangalore,2013-10-11T00:00:00.000Z
Chandini,35,Hyderabad,2015-10-17T00:00:00.000Z
Deepthi,40,Nasik,2022-12-21T00:00:00.000Z
Swapna,37,Mumbai,2021-04-14T00:00:00.000Z
Tharun,25,Delhi,2021-06-26T00:00:00.000Z


root
 |-- emp_name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- City: string (nullable = true)
 |-- start_time: timestamp (nullable = true)



In [0]:
df_dt_add_ts = (df_dt_ts
  # add a new column that adds 12 months to each date in the "start_date" column
  .withColumn("next_year_date", add_months("start_time", 12))
  # add a new column that adds 3 months to each date in the "start_date" column
  .withColumn("after_3_months", add_months("start_time", 3))
  # add a new column that subtract 12 months to each date in the "start_date" column
  .withColumn("before_one_year", add_months("start_time", -12))
  # add a new column that subtract 6 months to each date in the "start_date" column
  .withColumn("before_6_months", add_months("start_time", -6)))

display(df_dt_add_ts)

emp_name,Age,City,start_time,next_year_date,after_3_months,before_one_year,before_6_months
Sowmya,29,Chennai,2020-10-25T00:00:00.000Z,2021-10-25,2021-01-25,2019-10-25,2020-04-25
Bole,32,Bangalore,2013-10-11T00:00:00.000Z,2014-10-11,2014-01-11,2012-10-11,2013-04-11
Chandini,35,Hyderabad,2015-10-17T00:00:00.000Z,2016-10-17,2016-01-17,2014-10-17,2015-04-17
Deepthi,40,Nasik,2022-12-21T00:00:00.000Z,2023-12-21,2023-03-21,2021-12-21,2022-06-21
Swapna,37,Mumbai,2021-04-14T00:00:00.000Z,2022-04-14,2021-07-14,2020-04-14,2020-10-14
Tharun,25,Delhi,2021-06-26T00:00:00.000Z,2022-06-26,2021-09-26,2020-06-26,2020-12-26


##### 4) Using date_add() (365 Days)
- But this is `not fully accurate` because `leap years have 366 days`.
- `Best practice`: Use `add_months(12)`.

In [0]:
from pyspark.sql.functions import date_add

df_days = df_dt.withColumn("after_1_year", date_add("start_date", 365))
display(df_days)

emp_name,Age,City,start_date,after_1_year
Sowmya,29,Chennai,2020-10-25,2021-10-25
Bole,32,Bangalore,2013-10-11,2014-10-11
Chandini,35,Hyderabad,2015-10-17,2016-10-16
Deepthi,40,Nasik,2022-12-21,2023-12-21
Swapna,37,Mumbai,2021-04-14,2022-04-14
Tharun,25,Delhi,2021-06-26,2022-06-26


##### 5) Handle End-of-Month
- `add_months()` automatically `adjusts for month-end dates`.

In [0]:
data = [("2026-01-31",),
        ("2025-02-28",),
        ("2025-03-31",),
        ("2025-06-30",)]

df_eom = spark.createDataFrame(data, ["date"]) \
    .withColumn("date", col("date").cast("date")) \
    .withColumn("next_month", add_months("date", 1))

display(df_eom)

date,next_month
2026-01-31,2026-02-28
2025-02-28,2025-03-28
2025-03-31,2025-04-30
2025-06-30,2025-07-30


##### 6) Dynamic Months (Using Column Value)
- Months added based on another column value.

In [0]:
data = [
    ("2025-01-01", 2),
    ("2025-02-01", 1),
    ("2025-03-01", 0),
    ("2025-04-01", -1),
    ("2025-05-01", -2),
    ("2025-06-01", -3),
    ("2025-07-01", -4),
]

df_dynamic = spark.createDataFrame(data, ["start_date", "months_to_add"]) \
    .withColumn("start_date", col("start_date").cast("date")) \
    .withColumn("new_date", add_months("start_date", col("months_to_add")))

display(df_dynamic)

start_date,months_to_add,new_date
2025-01-01,2,2025-03-01
2025-02-01,1,2025-03-01
2025-03-01,0,2025-03-01
2025-04-01,-1,2025-03-01
2025-05-01,-2,2025-03-01
2025-06-01,-3,2025-03-01
2025-07-01,-4,2025-03-01


##### 7) Using SQL Query

In [0]:
df_dt.createOrReplaceTempView("orders")

spark.sql("""
SELECT 
    start_date,
    add_months(start_date, 12) AS after_1_year
FROM orders
""").display()

start_date,after_1_year
2020-10-25,2021-10-25
2013-10-11,2014-10-11
2015-10-17,2016-10-17
2022-12-21,2023-12-21
2021-04-14,2022-04-14
2021-06-26,2022-06-26
